# MIPIC on Kaggle

TinyBERT-6L, paper Table 7 settings: 5 epochs, lr `2e-5`, max length `256`,
**global batch 16**, cosine + 10% warmup, AdamW, `tau = 0.05`, `alpha = 0.4`,
PIC chain `(1,16) -> (2,32) -> (3,64) -> (4,256) -> (5,512) -> (6,768)`.
Corpus: the supplied `final_data.csv`, used whole (all 20,240 rows, `DGEM_FORMAT` kept).

**Run cells top to bottom.** Cell 6 picks the execution mode — read it before running.

| | |
| --- | --- |
| `RUN_MODE = "single"` | 1 GPU, per-device batch 16. Global batch 16 = paper-exact. No NCCL, so it cannot deadlock. **Default.** |
| `RUN_MODE = "ddp"` | 2xT4, 8/GPU x 2 = global batch 16. Faster, but this is the path that hung. |

Both modes write the same checkpoint and resume from each other, because both
produce 1,265 optimizer steps per epoch (20,240/16 on one rank; 10,120/8 on each
of two ranks). The scheduler state is therefore interchangeable.

## 1. Repository bootstrap

In [ ]:
from pathlib import Path
import os, sys, shutil, zipfile

work_root = Path("/kaggle/working")
current = Path.cwd()

if (current / "run_experiments.py").exists():
    repo_root = current
else:
    repo_root = work_root / "mipic"
    candidates = list(Path("/kaggle/input").rglob("run_experiments.py")) if Path("/kaggle/input").exists() else []
    if candidates:
        source = candidates[0].parent
        if repo_root.exists():
            shutil.rmtree(repo_root)
        shutil.copytree(source, repo_root,
                        ignore=shutil.ignore_patterns("results", "__pycache__", ".pytest_cache", "*.pyc"))
    else:
        zips = []
        for candidate in (Path("/kaggle/input").rglob("*.zip") if Path("/kaggle/input").exists() else []):
            try:
                with zipfile.ZipFile(candidate) as zf:
                    if any(n.endswith("run_experiments.py") for n in zf.namelist()):
                        zips.append(candidate)
            except zipfile.BadZipFile:
                continue
        if not zips:
            raise FileNotFoundError("MIPIC repository not found under /kaggle/input")
        if repo_root.exists():
            shutil.rmtree(repo_root)
        repo_root.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zips[0]) as zf:
            zf.extractall(repo_root)
        repo_root = next(iter(repo_root.rglob("run_experiments.py"))).parent

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root)
print("Repository:", repo_root)
version_file = repo_root / "VERSION.txt"
if version_file.exists():
    print("Version:", version_file.read_text(encoding="utf-8").splitlines()[0])
else:
    print("Version: VERSION.txt not present; using repository files as uploaded.")

Repository: /kaggle/working/mipic
Version: MIPIC_FINAL_V20


In [ ]:
mipic_source = (repo_root / "models" / "mrl_sia_pic.py").read_text(encoding="utf-8")
mipic_kl_disabled = "use_attention_kl=False" in mipic_source
print("MIPIC KL disabled:", mipic_kl_disabled)
assert mipic_kl_disabled, (
    "Uploaded MIPIC code still enables KL. Upload the modified "
    "models/mrl_sia_pic.py and restart the Kaggle session."
 )

In [ ]:
%pip install -q -r requirements-kaggle.txt

## 2. Hardware

In [3]:
import torch
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(" ", i, torch.cuda.get_device_name(i))
assert torch.cuda.is_available(), "Enable a GPU accelerator (Settings -> Accelerator)."

PyTorch: 2.10.0+cu128 | CUDA: True | GPUs: 2
  0 Tesla T4
  1 Tesla T4


## 3. Locate the dataset

Nothing is filtered or regenerated. Do **not** run `prepare_training_corpus.py`.

In [4]:
import pandas as pd
from pathlib import Path

matches = sorted(Path("/kaggle/input").rglob("final_data.csv"))
if not matches:
    raise FileNotFoundError("Attach the Kaggle dataset containing final_data.csv (Add Input).")

# Point at the DIRECTORY. run_experiments.py resolves the three files it needs
# from it: final_data.csv (paper A.2 pooled corpus), tweet_train_3k.csv and
# tweet_test.csv (Table 6 probe splits). There is one configuration, no presets.
data_dir = matches[0].parent
os.environ["MIPIC_DATA_DIR"] = str(data_dir)
print("Data directory:", data_dir)

# tweet_train.csv is the PROBE train split (26,732 rows, full).
# tweet_train_3k.csv is the slice inside the contrastive corpus.
expected = ["final_data.csv", "tweet_train.csv", "tweet_test.csv", "tweet_train_3k.csv"]
for name in expected:
    found = data_dir / name
    if not found.exists():
        hits = sorted(Path("/kaggle/input").rglob(name)) or [Path(repo_root) / "data" / name]
        found = hits[0]
    status = "ok " if found.exists() else "MISSING"
    rows = len(pd.read_csv(found)) if found.exists() else 0
    print(f"  {status} {name:22} {rows:>6} rows   {found}")

frame = pd.read_csv(data_dir / "final_data.csv")
print("\nPooled corpus sources:")
print(frame["source"].value_counts().to_string())


Data directory: /kaggle/input/datasets/manojanaparthi/mipic-a1-kag/mipic/data
  ok  final_data.csv          20240 rows   /kaggle/input/datasets/manojanaparthi/mipic-a1-kag/mipic/data/final_data.csv
  ok  tweet_train.csv         26732 rows   /kaggle/input/datasets/manojanaparthi/mipic-a1-kag/mipic/data/tweet_train.csv
  ok  tweet_test.csv           3432 rows   /kaggle/input/datasets/manojanaparthi/mipic-a1-kag/mipic/data/tweet_test.csv
  ok  tweet_train_3k.csv       3000 rows   /kaggle/input/datasets/manojanaparthi/mipic-a1-kag/mipic/data/tweet_train_3k.csv

Pooled corpus sources:
source
BANKING77          3000
TWEET_EVAL_SENT    3000
DEEHAN1866/WIC     3000
MRPC               3000
DGEM_FORMAT        3000
STSB               2893
SICK               2347


## 4. Test gate

Never spend GPU quota before this passes. Expect **100 passed**.

In [5]:
!python -m compileall -q . && echo "compileall OK"
!python -m pytest -q

compileall OK
........................................................................ [ 60%]
................................................                         [100%]
=============================== warnings summary ===============================
tests/test_mipic_integration.py::test_1_and_2_model_construction_and_alpha_default
  <frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyPacked has no __module__ attribute

tests/test_mipic_integration.py::test_1_and_2_model_construction_and_alpha_default
  <frozen importlib._bootstrap>:488: DeprecationWarning: builtin type SwigPyObject has no __module__ attribute

tests/test_mipic_integration.py::test_1_and_2_model_construction_and_alpha_default
tests/test_mrl_baseline_integration.py::test_refactored_mrl_matches_uploaded_baseline_and_trains_one_step
tests/test_mrl_pic_integration.py::test_1_and_2_model_construction_and_tinybert_specs
tests/test_mrl_sia_integration.py::test_1_tinybert_returns_seven_hidden_states
t

## 5. Shape and gradient traces (CPU, seconds)

In [6]:
!python tensor_traces.py


MRL trace  --  Eq. (1) x Eq. (19):  B=2, D=8, nested dims [2, 4, 8]
view_a (h_CLS of view 1)      shape (2, 8)
view_b (h_CLS of view 2)      shape (2, 8)
  d=2  prefix slice (2, 2) -> normalize (2, 2) -> logits (2, 2) (= q k^T / tau, tau=0.05)  labels [0, 1]  CE=0.0116
  d=4  prefix slice (2, 4) -> normalize (2, 4) -> logits (2, 2) (= q k^T / tau, tau=0.05)  labels [0, 1]  CE=0.1222
  d=8  prefix slice (2, 8) -> normalize (2, 8) -> logits (2, 2) (= q k^T / tau, tau=0.05)  labels [0, 1]  CE=5.9371
  sum over prefixes (Eq. 1, unweighted) = 6.0709   (manual 6.0709)
  returned logit keys: ['dim_2', 'dim_4', 'dim_8']
  reduction: cross_entropy default 'mean' over the batch, then summed over d
  grad path -> view_a (2, 8) |g|=5.8207
  grad path -> view_b (2, 8) |g|=4.2610
  under DDP: keys are all_gathered (differentiable), labels become
             arange(rank*B, (rank+1)*B) so the diagonal stays the positive

SIA trace  --  Eq. (4)-(13):  B=2, L=5, D=8, prefix d_i=4
hidden (layer k)     

## 6. Execution mode and run configuration

`RUN_MODE = "single"` is the default: one GPU at per-device batch 16 is the same
global batch 16 the paper specifies, and it removes NCCL entirely.

Switch to `"ddp"` only if you want the ~2x speedup and are willing to hit the
hang again. The DDP mitigations below are all applied automatically in that mode.

In [ ]:
RUN_MODE = "single"          # "single" | "ddp"
OUTPUT_DIR = "/kaggle/working/mipic_results_no_kl"
FRESH = True                 # Start a new no-KL experiment

os.environ["MIPIC_OUTPUT_DIR"] = OUTPUT_DIR
os.environ["MIPIC_FRESH"] = "1" if FRESH else "0"
os.environ["MIPIC_EPOCHS"] = "5"
os.environ["MIPIC_LEARNING_RATE"] = "2e-5"
os.environ["MIPIC_MAX_LENGTH"] = "256"
os.environ["MIPIC_AMP"] = "1"

# Use zero workers for predictable Kaggle execution.
os.environ["MIPIC_NUM_WORKERS"] = "0"
os.environ["MIPIC_LOG_EVERY"] = "50"
os.environ["MIPIC_STACK_DUMP_SECONDS"] = "900"

if RUN_MODE == "ddp":
    os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
    os.environ["MIPIC_BATCH_SIZE"] = "8"
    os.environ["MIPIC_DDP_TIMEOUT_MIN"] = "20"
    os.environ["NCCL_P2P_DISABLE"] = "1"
    os.environ["MIPIC_DDP_STATIC_GRAPH"] = "1"
    LAUNCH = "torchrun --standalone --nproc_per_node=2 run_experiments.py"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    os.environ["MIPIC_BATCH_SIZE"] = "16"
    os.environ.pop("MIPIC_DDP_STATIC_GRAPH", None)
    os.environ.pop("MIPIC_DDP_TIMEOUT_MIN", None)
    os.environ.pop("NCCL_P2P_DISABLE", None)
    LAUNCH = "python run_experiments.py"

import subprocess
visible = int(subprocess.run(
    [sys.executable, "-c", "import torch; print(torch.cuda.device_count())"],
    capture_output=True, text=True, env=os.environ,
 ).stdout.strip() or 0)
need = 2 if RUN_MODE == "ddp" else 1
assert visible >= need, (
    f"RUN_MODE={RUN_MODE!r} needs {need} GPU(s); subprocess sees {visible}. "
    "Enable the required GPU accelerator and restart the session."
 )

print("mode:", RUN_MODE)
print("GPUs visible:", visible)
print("per-device batch:", os.environ["MIPIC_BATCH_SIZE"],
      "| global batch:", int(os.environ["MIPIC_BATCH_SIZE"]) * (2 if RUN_MODE == "ddp" else 1))
print("output:", OUTPUT_DIR, "| fresh:", FRESH)
print("launch:", LAUNCH)

mode: ddp
CUDA_VISIBLE_DEVICES: 0,1 -> subprocess sees 2 GPU(s)
per-device batch: 8 | global batch: 16
output: /kaggle/working/mipic_results | fresh: False
launch: torchrun --standalone --nproc_per_node=2 run_experiments.py


## 7. Resume status

Completed epochs are never lost. Check before launching.

In [8]:
state_path = Path(OUTPUT_DIR) / "checkpoints" / "mipic" / "training_state.pt"
done_path  = Path(OUTPUT_DIR) / "checkpoints" / "mipic" / "model.pt"
if state_path.exists():
    saved = torch.load(state_path, map_location="cpu")
    print(f"Resumable checkpoint: epoch {saved['epoch']} of 5")
    for i, h in enumerate(saved.get("history", []), 1):
        print("  epoch", i, {k: round(v, 4) for k, v in h.items()})
    if FRESH:
        print("\n!! FRESH=True will DELETE this. Set FRESH=False and re-run cell 6.")
elif done_path.exists():
    print("Completed checkpoint present:", done_path, done_path.stat().st_size, "bytes")
else:
    print("No checkpoint; will start from scratch.")

No checkpoint; will start from scratch.


## 8. Pipeline check (about a minute) - NOT a result

Truncated training that still exercises `train -> save -> assert non-empty ->
evaluate`, so a path or checkpoint problem surfaces now rather than after hours
of GPU time.

It trains on **48 sentences** (3 batches x 8/rank x 2 ranks), i.e. 0.24% of one
epoch, and probes 300/300 rows. The run prints a `TRUNCATED RUN` banner and a
`PLUMBING CHECK ONLY` warning over its results table, and stamps
`truncated_run: true` into its results JSON. **Ignore the numbers.** Writes to a
throwaway directory.

In [9]:
check_env = {
    "MIPIC_MAX_BATCHES": "3",
    "MIPIC_SMOKE_FULL_PIPELINE": "1",
    "MIPIC_EPOCHS": "1",
    "MIPIC_EVAL_TRAIN_SAMPLES": "300",
    "MIPIC_EVAL_TEST_SAMPLES": "300",
    "MIPIC_FRESH": "1",
    "MIPIC_OUTPUT_DIR": "/kaggle/working/mipic_pipeline_check",
}
prefix = " ".join(f"{k}={v}" for k, v in check_env.items())
!{prefix} {LAUNCH} --model mipic

W0906 11:04:40.141000 203 torch/distributed/run.py:852] 
W0906 11:04:40.141000 203 torch/distributed/run.py:852] *****************************************
W0906 11:04:40.141000 203 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0906 11:04:40.141000 203 torch/distributed/run.py:852] *****************************************
[W906 11:04:40.043154968 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W906 11:04:46.370209143 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W906 11:04:46.436517715 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
DDP: nccl, world_size=2, collective timeout=20 min, NCCL_P2P_DISABLE=1, NCCL_SHM_DISABLE=<unset>
Seed: 42 (rank-adjusted per process; this rank uses 42

## 9. Full 5-epoch run - this is the real one

Heartbeats look like `[rank 0] 22:31:04 epoch 3/5: batch 550/1265`. Confirm you
see **`epoch 1/5`** and rising batch counts, and that no `TRUNCATED RUN` banner
appears. Expect ~13 minutes per arm on 2xT4.

If output stops for more than ~20 minutes, go to cell 11 **before** interrupting.

Each epoch prints a held-out probe score:

```
Epoch 1: held-out probe macro-F1 63.10  <- best so far, checkpointed
Epoch 2: held-out probe macro-F1 64.80  <- best so far, checkpointed
Epoch 3: held-out probe macro-F1 61.20
```

That probe is fitted on 80% of the probe-TRAIN rows and scored on the held-out
20%; the test split is never touched. The best epoch is kept in
`checkpoints/<arm>/best.pt` and evaluated alongside the final epoch, which
appears as `<arm>_last_epoch` in the results.

In [10]:
# !{LAUNCH} --model mrl

## 10. Results

In [11]:
import json
from pathlib import Path

results = Path(OUTPUT_DIR) / "tweeteval_results.json"
if results.exists():
    data = json.loads(results.read_text())
    print(json.dumps(data.get("metadata", {}), indent=1))
    for name, history in data.get("training_history", {}).items():
        print("\n" + name, "training history:")
        for i, h in enumerate(history, 1):
            print("  epoch", i, {k: round(v, 4) for k, v in h.items()})
    rows = []
    for name, dims in data.get("evaluations", {}).items():
        for dim, m in sorted(dims.items(), key=lambda kv: int(kv[0])):
            rows.append({"Model": name, "Dim": int(dim),
                         "Acc": round(m["accuracy"] * 100, 2),
                         "Macro_F1": round(m["f1"] * 100, 2),
                         "probe_iters": m.get("probe_n_iter"),
                         "converged": m.get("probe_converged")})
    display(pd.DataFrame(rows))
    if data.get("evaluation_errors"):
        print("\nEVALUATION ERRORS:", data["evaluation_errors"])
else:
    print("No results yet:", results)

for p in sorted(Path(OUTPUT_DIR).rglob("*")):
    if p.is_file():
        print(f"{p}  ({p.stat().st_size:,} bytes)")

No results yet: /kaggle/working/mipic_results/tweeteval_results.json


## 11. If the run hangs

Output stops mid-epoch, both processes alive, near-zero CPU/GPU.

`py-spy` will **not** work here — Kaggle's container blocks `ptrace`
(`Permission denied (os error 13)`). The cell below reads `/proc` instead, which
is always permitted. Run it from a **second notebook session** while the run is
still stuck, or from this one if the training cell is in a different session.

In [12]:
# !python diagnose_hang.py --sample-seconds 5

Read the output:

| Observation | Diagnosis | Action |
| --- | --- | --- |
| Every thread ~0% CPU, state `S`, WCHAN `futex_wait` / `pipe_read` | waiting on another process | `MIPIC_NUM_WORKERS=0` (already default), then `NCCL_SHM_DISABLE=1` |
| Threads pinned near 100%, state `R` | NCCL collective spinning for a peer that never posted | `MIPIC_DDP_STATIC_GRAPH=1` (already default in ddp mode), then `NCCL_SHM_DISABLE=1` |
| State `D`, low disk in the DISK section | blocked on I/O | free space in `/kaggle/working` |
| Only one process listed | the peer rank died | scroll the training log up for its exception |

Then set `RUN_MODE = "single"` in cell 6, confirm cell 7 shows the resumable
epoch, and re-run cell 9. Single-GPU mode is paper-exact on global batch size
and cannot deadlock.

## 12. Evaluation only, from an explicit checkpoint

Path-independent; does not require a training run in the same session.

In [13]:
ckpt = Path(OUTPUT_DIR) / "checkpoints" / "mipic" / "model.pt"
if ckpt.exists():
    !MIPIC_CHECKPOINT={ckpt} python run_experiments.py --model mipic --eval-only
else:
    print("No completed checkpoint at", ckpt)

No completed checkpoint at /kaggle/working/mipic_results/checkpoints/mipic/model.pt


## 13. Optional: all four controlled conditions

`mrl`, `mrl_sia`, `mrl_pic`, `mipic` — identical corpus, batch size, schedule,
clipping and representation, so the arms are directly comparable. Roughly 4x the
runtime. `--skip-existing` resumes a partial sweep.

In [14]:
# MIPIC_OUTPUT_DIR=/kaggle/working/mipic_results_all
!MIPIC_OUTPUT_DIR=/kaggle/working/mipic_results_all {LAUNCH} --model all --skip-existing

W0906 11:05:08.589000 258 torch/distributed/run.py:852] 
W0906 11:05:08.589000 258 torch/distributed/run.py:852] *****************************************
W0906 11:05:08.589000 258 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0906 11:05:08.589000 258 torch/distributed/run.py:852] *****************************************
[W906 11:05:08.407178883 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W906 11:05:15.784195196 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W906 11:05:15.864029628 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
DDP: nccl, world_size=2, collective timeout=20 min, NCCL_P2P_DISABLE=1, NCCL_SHM_DISABLE=<unset>
Seed: 42 (rank-adjusted per process; this rank uses 42

## 14. POS dimension selection (fixed budgets)

This compares prefix probes using the first k coordinates with learned selectors that have **exactly k active coordinates on every training and evaluation forward pass**. It runs k = 16, 32, 64, 128, and 256. The MIPIC encoder is frozen; only each selector and linear POS classifier train. The selector uses the train split, validation selects the checkpoint, and the test split is evaluated once. Both probes use the same train-only feature normalization.

Results are saved to `/kaggle/working/mipic_pos_probe_budget_sweep/pos_probe_budget_results.json` and `.csv`. This does not retrain MIPIC.

In [ ]:
from pathlib import Path

POS_CHECKPOINT = Path(
    "/kaggle/input/datasets/maverick0215/output/"
    "mipic_results_all/checkpoints/mipic/model.pt"
)
POS_SCRIPT = Path(
    "/kaggle/input/datasets/maverick0215/mipic-implementation/"
    "MIPIC_LEXIS_A1/pos_probe.py"
)
POS_OUTPUT_DIR = Path("/kaggle/working/mipic_pos_probe_budget_sweep")

if not POS_CHECKPOINT.is_file():
    raise FileNotFoundError(f"MIPIC checkpoint not found: {POS_CHECKPOINT}")
if not POS_SCRIPT.is_file():
    raise FileNotFoundError(f"POS probe script not found: {POS_SCRIPT}")

print("Using final MIPIC checkpoint:", POS_CHECKPOINT)
print("Testing prefix and learned selectors at k=16, 32, 64, 128, 256.")
!python "{POS_SCRIPT}" --checkpoint "{POS_CHECKPOINT}" --output-dir "{POS_OUTPUT_DIR}" --k-values 16,32,64,128,256

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

POS_CHECKPOINT = Path("/kaggle/input/datasets/maverick0215/output/mipic_results_all/checkpoints/mipic/model.pt")
POS_SCRIPT = Path("/kaggle/input/datasets/maverick0215/mipic-implementation/MIPIC_LEXIS_A1/pos_probe_spars.py")
POS_OUTPUT_DIR = Path("/kaggle/working/mipic_pos_probe_spars")

if not POS_CHECKPOINT.is_file():
    raise FileNotFoundError(f"MIPIC checkpoint not found: {POS_CHECKPOINT}")
if not POS_SCRIPT.is_file():
    raise FileNotFoundError(f"Sparse-gate POS probe script not found: {POS_SCRIPT}")

print("Using final MIPIC checkpoint:", POS_CHECKPOINT)
print("Running sparse learned gates with budgets k=16, 32, 64, 128, 256.")
subprocess.run([
    sys.executable, str(POS_SCRIPT),
    "--checkpoint", str(POS_CHECKPOINT),
    "--output-dir", str(POS_OUTPUT_DIR),
    "--k-values", "16,32,64,128,256",
], check=True)

result_path = POS_OUTPUT_DIR / "pos_probe_spars_results.json"
report = json.loads(result_path.read_text(encoding="utf-8"))
rows = []
for k, metrics in report["summary"].items():
    row = {"k": int(k)}
    for label, key in [("Prefix", "prefix_test_macro_f1"), ("Sparse gates", "learned_test_macro_f1")]:
        row[f"{label} macro-F1 mean"] = metrics[key]["mean"]
        row[f"{label} macro-F1 std"] = metrics[key]["std"]
    row["Mean selected gate value"] = metrics["selected_gate_value"]["mean"]
    row["Selected in first k"] = metrics["selected_fraction_in_first_k"]["mean"]
    row["Pairwise Jaccard"] = metrics["mean_pairwise_jaccard"]
    rows.append(row)
display(pd.DataFrame(rows).sort_values("k").reset_index(drop=True))
print("Full per-seed results:", result_path)


## 15. Dense POS gates with L2 regularization

This probe learns one sigmoid scale for every one of MIPIC's 768 token-vector dimensions. All coordinates remain present; there is no top-k or active-dimension budget. The training objective is POS cross-entropy plus `GATE_L2 * sum(gate**2)`. The penalty encourages smaller gates but does not guarantee that they fall below 0.5. A full-width, ungated linear POS probe is included as a control. The encoder stays frozen; validation macro-F1 selects each probe checkpoint, and test metrics are reported for that checkpoint.

The cell prints test scores and gate summaries, displays each dimension's mean learned gate across seeds, and writes JSON/CSV files to `/kaggle/working/mipic_pos_probe_l2/`. Gate magnitudes are not causal feature importance: a linear classifier can partly compensate for gate scaling in its own weights. Interpret gate values together with the regularization strength and test performance.

In [ ]:
from pathlib import Path
import json, subprocess, sys
import pandas as pd

POS_CHECKPOINT = Path("/kaggle/input/datasets/maverick0215/output/mipic_results_all/checkpoints/mipic/model.pt")
POS_SCRIPT = Path("/kaggle/input/datasets/maverick0215/mipic-implementation/MIPIC_LEXIS_A1/pos_probe_l2.py")
POS_OUTPUT_DIR = Path("/kaggle/working/mipic_pos_probe_l2")
GATE_L2 = 1e-3

if not POS_CHECKPOINT.is_file():
    raise FileNotFoundError(f"MIPIC checkpoint not found: {POS_CHECKPOINT}")
if not POS_SCRIPT.is_file():
    raise FileNotFoundError(f"L2 POS probe script not found: {POS_SCRIPT}")

print("Using frozen MIPIC checkpoint:", POS_CHECKPOINT)
print("Training dense gates over all 768 dimensions; L2 coefficient:", GATE_L2)
subprocess.run([
    sys.executable, str(POS_SCRIPT),
    "--checkpoint", str(POS_CHECKPOINT),
    "--output-dir", str(POS_OUTPUT_DIR),
    "--gate-l2", str(GATE_L2),
], check=True)

summary_path = POS_OUTPUT_DIR / "pos_probe_l2_results.json"
report = json.loads(summary_path.read_text(encoding="utf-8"))
rows = []
for label, key in [("Full-width baseline", "baseline_test_macro_f1"), ("L2-gated", "gated_test_macro_f1")]:
    metric = report["summary_mean_std"][key]
    rows.append({"Probe": label, "Test macro-F1 mean": metric["mean"], "Test macro-F1 std": metric["std"]})
for label, key in [("Full-width baseline", "baseline_test_accuracy"), ("L2-gated", "gated_test_accuracy")]:
    metric = report["summary_mean_std"][key]
    for row in rows:
        if row["Probe"] == label:
            row["Test accuracy mean"] = metric["mean"]
            row["Test accuracy std"] = metric["std"]
display(pd.DataFrame(rows))

for key in ["gate_mean", "gate_std", "gate_min", "gate_max", "fraction_gate_below_0_5", "fraction_gate_below_0_25", "first_128_gate_mean", "remaining_gate_mean", "dimension_index_spearman_rho"]:
    stats = report["summary_mean_std"].get(key)
    if stats:
        print(f"{key}: {stats['mean']:.4f} ± {stats['std']:.4f}")

prefix_path = POS_OUTPUT_DIR / "pos_probe_l2_prefix_gate_means.csv"
print("Mean gate weight for each initial prefix (mean ± std across seeds):")
display(pd.read_csv(prefix_path))

gates_path = POS_OUTPUT_DIR / "pos_probe_l2_dimension_gates.csv"
gates = pd.read_csv(gates_path)
print("Per-dimension gate means and across-seed variation:")
display(gates)
print("Full report:", summary_path)
print("Per-seed summary:", POS_OUTPUT_DIR / "pos_probe_l2_summary.csv")
print("Per-dimension gate weights:", gates_path)